# MedTrack_DV – Milestone 1
## Notebook 05: KPI Engineering
**Purpose:** Calculate all 6 mandatory KPIs and build the 4 final analytical datasets for Tableau dashboards

### 6 Mandatory KPIs:
1. Total Admissions
2. Occupancy Rate
3. Average Length of Stay
4. Readmission Rate
5. Bed Utilization Rate
6. Department Efficiency Score

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PROCESSED = '../data/processed/'

df_patients      = pd.read_csv(PROCESSED + 'normalized_patients.csv')
df_bed_records   = pd.read_csv(PROCESSED + 'normalized_bed_records.csv', parse_dates=['admission_date','discharge_date'])
df_department    = pd.read_csv(PROCESSED + 'normalized_departments.csv')
df_doctor        = pd.read_csv(PROCESSED + 'normalized_doctors.csv')
df_nurse         = pd.read_csv(PROCESSED + 'normalized_nurses.csv')
df_ward          = pd.read_csv(PROCESSED + 'normalized_wards.csv')
df_beds_patients = pd.read_csv(PROCESSED + 'normalized_beds_patients.csv')
df_beds_services = pd.read_csv(PROCESSED + 'normalized_beds_services.csv')
df_readmission   = pd.read_csv(PROCESSED + 'normalized_readmission.csv')
df_healthcare    = pd.read_csv(PROCESSED + 'normalized_healthcare.csv', parse_dates=['date_of_admission','discharge_date'])

print('All normalized datasets loaded!')

## KPI 1: Total Admissions

In [ ]:
# Formula: COUNT DISTINCT admission_id
total_admissions_hmis = df_bed_records['admission_id'].nunique()
total_admissions_hc   = len(df_healthcare)
total_admissions_re   = df_readmission['patient_id'].nunique()

print('=== KPI 1: TOTAL ADMISSIONS ===')
print(f'HMIS BedRecords:    {total_admissions_hmis:,}')
print(f'Healthcare Dataset: {total_admissions_hc:,}')
print(f'Readmission Data:   {total_admissions_re:,}')
print(f'Combined Total:     {total_admissions_hmis + total_admissions_hc:,}')

## KPI 2: Occupancy Rate

In [ ]:
# Formula: (Occupied Beds / Total Beds) x 100
total_beds      = 500  # From HMIS Bed table
occupied_beds   = df_bed_records['bed_no'].nunique() if 'bed_no' in df_bed_records.columns else df_beds_services['patients_admitted'].mean()
occupancy_rate  = round((occupied_beds / total_beds) * 100, 2)

# From Beds Services dataset
avg_util_from_beds = df_beds_services['patients_admitted'].sum() / df_beds_services['available_beds'].sum() * 100

print('=== KPI 2: OCCUPANCY RATE ===')
print(f'Total Beds (HMIS):         {total_beds}')
print(f'Occupancy Rate (HMIS):     {occupancy_rate}%')
print(f'Avg Bed Util (Beds Mgmt):  {avg_util_from_beds:.2f}%')

## KPI 3: Average Length of Stay

In [ ]:
# Formula: AVG(discharge_date - admission_date) in days
avg_los_hmis = df_bed_records['length_of_stay_days'].dropna().mean()
avg_los_hc   = df_healthcare['length_of_stay_days'].dropna().mean()

duration_col = None
for col in ['duration_of_stay', 'length_of_stay_days']:
    if col in df_readmission.columns:
        duration_col = col
        break
avg_los_re = df_readmission[duration_col].dropna().mean() if duration_col else 'N/A'

print('=== KPI 3: AVERAGE LENGTH OF STAY ===')
print(f'HMIS BedRecords:    {avg_los_hmis:.2f} days')
print(f'Healthcare Dataset: {avg_los_hc:.2f} days')
print(f'Readmission Data:   {avg_los_re:.2f} days' if isinstance(avg_los_re, float) else f'Readmission Data: {avg_los_re}')

## KPI 4: Readmission Rate

In [ ]:
# Formula: (Readmitted Patients / Total Eligible Patients) x 100
# Using Readmission dataset: DAMA or EXPIRY outcome = readmission risk
total_eligible   = len(df_readmission)
readmitted       = df_readmission['outcome'].str.strip().str.upper().isin(['DAMA', 'EXPIRY']).sum() if 'outcome' in df_readmission.columns else 0
readmission_rate = round((readmitted / total_eligible) * 100, 2) if total_eligible > 0 else 0

print('=== KPI 4: READMISSION RATE ===')
print(f'Total Patients:     {total_eligible:,}')
print(f'Readmitted:         {readmitted:,}')
print(f'Readmission Rate:   {readmission_rate}%')

## KPI 5: Bed Utilization Rate

In [ ]:
# Formula: (Beds in Use / Total Available Beds) x 100
total_available  = df_beds_services['available_beds'].sum()
total_admitted   = df_beds_services['patients_admitted'].sum()
bed_util_rate    = round((total_admitted / total_available) * 100, 2)

# Weekly breakdown
weekly_util = df_beds_services.groupby('service').apply(
    lambda x: round((x['patients_admitted'].sum() / x['available_beds'].sum()) * 100, 2)
).reset_index()
weekly_util.columns = ['service', 'bed_utilization_rate_%']

print('=== KPI 5: BED UTILIZATION RATE ===')
print(f'Total Available Beds: {total_available:,}')
print(f'Total Admitted:       {total_admitted:,}')
print(f'Overall Utilization:  {bed_util_rate}%')
print('\nBy Service/Department:')
display(weekly_util)

## KPI 6: Department Efficiency Score

In [ ]:
# Formula: Composite score using Occupancy + LOS + Satisfaction (normalized 0-100)
dept_efficiency = df_beds_services.groupby('service').agg(
    total_admitted   = ('patients_admitted', 'sum'),
    total_refused    = ('patients_refused', 'sum'),
    available_beds   = ('available_beds', 'mean'),
    avg_satisfaction = ('patient_satisfaction', 'mean'),
    avg_staff_morale = ('staff_morale', 'mean')
).reset_index()

# Occupancy component
dept_efficiency['occupancy_score'] = (
    dept_efficiency['total_admitted'] /
    (dept_efficiency['total_admitted'] + dept_efficiency['total_refused']) * 100
).round(2)

# Satisfaction component (normalize to 0-100)
dept_efficiency['satisfaction_score'] = (
    dept_efficiency['avg_satisfaction'] / dept_efficiency['avg_satisfaction'].max() * 100
).round(2)

# Staff morale component
dept_efficiency['staff_score'] = (
    dept_efficiency['avg_staff_morale'] / dept_efficiency['avg_staff_morale'].max() * 100
).round(2)

# Final score: weighted average (40% occupancy, 40% satisfaction, 20% staff)
dept_efficiency['department_efficiency_score'] = (
    0.40 * dept_efficiency['occupancy_score'] +
    0.40 * dept_efficiency['satisfaction_score'] +
    0.20 * dept_efficiency['staff_score']
).round(2)

print('=== KPI 6: DEPARTMENT EFFICIENCY SCORE ===')
print('Formula: 40% Occupancy + 40% Satisfaction + 20% Staff Morale (Scale: 0-100)')
display(dept_efficiency[['service','occupancy_score','satisfaction_score','staff_score','department_efficiency_score']].sort_values('department_efficiency_score', ascending=False))

## Build Final 4 Analytical Datasets for Tableau

In [ ]:
# =============================================
# DATASET 1: hospital_overview_dataset.csv
# Grain: One row = One admission
# =============================================
hospital_overview = df_bed_records.merge(
    df_patients[['patient_id','gender','date_of_birth','full_name']],
    on='patient_id', how='left'
).merge(
    df_ward[['ward_no','ward_name','dept_id']],
    on='ward_no', how='left'
).merge(
    df_department[['dept_id','dept_name']],
    on='dept_id', how='left'
)

hospital_overview['hospital_id']   = 'H001'
hospital_overview['hospital_name'] = 'MedTrack General Hospital'

hospital_overview.to_csv(PROCESSED + 'hospital_overview_dataset.csv', index=False)
print(f'hospital_overview_dataset.csv saved: {hospital_overview.shape}')

In [ ]:
# =============================================
# DATASET 2: patient_flow_dataset.csv
# Grain: One row = One patient movement / admission event
# =============================================
patient_flow = df_healthcare[[
    'name','age','gender','medical_condition','date_of_admission',
    'discharge_date','admission_type','hospital','doctor',
    'billing_amount','length_of_stay_days','admission_year',
    'admission_month','test_results'
]].copy()

patient_flow['date_of_admission'] = pd.to_datetime(patient_flow['date_of_admission'])
patient_flow['discharge_date']    = pd.to_datetime(patient_flow['discharge_date'])
patient_flow['day_of_week']       = patient_flow['date_of_admission'].dt.day_name()
patient_flow['admission_quarter'] = patient_flow['date_of_admission'].dt.quarter
patient_flow['is_weekend']        = patient_flow['date_of_admission'].dt.weekday >= 5

patient_flow.to_csv(PROCESSED + 'patient_flow_dataset.csv', index=False)
print(f'patient_flow_dataset.csv saved: {patient_flow.shape}')

In [ ]:
# =============================================
# DATASET 3: department_analytics_dataset.csv
# Grain: One row = One service/department per week
# =============================================
dept_analytics = df_beds_services.merge(
    dept_efficiency[['service','occupancy_score','satisfaction_score','department_efficiency_score']],
    on='service', how='left'
)

dept_analytics['hospital_id']   = 'H001'
dept_analytics['hospital_name'] = 'MedTrack General Hospital'
dept_analytics['bed_utilization_rate'] = (
    dept_analytics['patients_admitted'] / dept_analytics['available_beds'] * 100
).round(2)
dept_analytics['refusal_rate'] = (
    dept_analytics['patients_refused'] /
    (dept_analytics['patients_admitted'] + dept_analytics['patients_refused']) * 100
).round(2)

dept_analytics.to_csv(PROCESSED + 'department_analytics_dataset.csv', index=False)
print(f'department_analytics_dataset.csv saved: {dept_analytics.shape}')

In [ ]:
# =============================================
# DATASET 4: resource_utilization_dataset.csv
# Grain: One row = One service/department per week (resource focus)
# =============================================
resource_util = df_beds_services.merge(
    df_beds_schedule.groupby(['week','service']).agg(
        total_staff       = ('staff_id', 'count'),
        staff_present     = ('present', 'sum')
    ).reset_index(),
    on=['week','service'], how='left'
)

resource_util['hospital_id']        = 'H001'
resource_util['hospital_name']      = 'MedTrack General Hospital'
resource_util['bed_utilization_pct'] = (
    resource_util['patients_admitted'] / resource_util['available_beds'] * 100
).round(2)
resource_util['staff_utilization_pct'] = (
    resource_util['staff_present'] / resource_util['total_staff'] * 100
).round(2)
resource_util['shortage_flag'] = resource_util['patients_refused'] > 0

resource_util.to_csv(PROCESSED + 'resource_utilization_dataset.csv', index=False)
print(f'resource_utilization_dataset.csv saved: {resource_util.shape}')

## Final KPI Summary

In [ ]:
print('=================================================')
print('   MEDTRACK_DV – FINAL KPI SUMMARY')
print('=================================================')
print(f'KPI 1 – Total Admissions:       {total_admissions_hmis + total_admissions_hc:,}')
print(f'KPI 2 – Occupancy Rate:         {occupancy_rate}%')
print(f'KPI 3 – Avg Length of Stay:     {avg_los_hmis:.2f} days (HMIS)')
print(f'KPI 4 – Readmission Rate:       {readmission_rate}%')
print(f'KPI 5 – Bed Utilization Rate:   {bed_util_rate}%')
print(f'KPI 6 – Dept Efficiency Score:  See department breakdown above')
print()
print('Final Datasets Created:')
print('  ✅ hospital_overview_dataset.csv')
print('  ✅ patient_flow_dataset.csv')
print('  ✅ department_analytics_dataset.csv')
print('  ✅ resource_utilization_dataset.csv')
print('=================================================')
print('Milestone 1 COMPLETE! Ready for Milestone 2.')